# License Plate Detection & Recognition — YOLOv8 + OCR

End-to-end ANPR (Automatic Number Plate Recognition) pipeline:

**YOLOv8** locates the plate in a photo/frame -> the plate region is **cropped**
and **preprocessed** -> **EasyOCR** reads the characters -> optional
**post-processing** cleans up the text.

### Before you run this
`Runtime > Change runtime type > T4 GPU`. Training on CPU will be very slow.

### Table of contents
1. Setup
2. Dataset
3. Dataset exploration
4. Training (YOLOv8)
5. Validation
6. Inference (detection only)
7. Cropping detected plates
8. OCR preprocessing
9. OCR with EasyOCR
10. End-to-end pipeline
11. Batch processing + CSV export
12. Bonus: video pipeline
13. Improving accuracy further
14. Export & persist the model

## 1. Setup

In [ ]:
!pip install -q ultralytics easyocr roboflow

In [ ]:
import os
import re
import cv2
import glob
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import easyocr
from IPython.display import Image as IPyImage, display

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
          "then Runtime > Restart session, and run this cell again.")

## 2. Dataset

**Recommended: "License Plate Recognition" (Roboflow Universe Projects)**
`universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e`

Why this one:
- 24,242 images total (21,174 train / 2,048 valid / 1,020 test) after 3x
  augmentation — enough to fine-tune a solid detector without a multi-hour download.
- Already labeled and exported in native YOLOv8 format (`data.yaml` + `train/valid/test`
  folders with `images/` and `labels/`), so there is no annotation conversion step.
- Single class (`License_Plate`), real-world traffic/parking photos with varied
  angles, lighting, and plate styles — generalizes better than a narrow regional set.
- CC BY 4.0 licensed, 700+ stars, one of the most-downloaded datasets in this exact
  category — this is the dataset most public YOLOv8+OCR plate tutorials use.

**Get a free API key**: create an account at `app.roboflow.com`, then
`Settings > Roboflow API` to copy your private key. Paste it below.

**No-signup alternative**: on the dataset page, click *Download Dataset* ->
choose the **YOLOv8** format -> download the zip directly, then upload it to
Colab (or Google Drive) and unzip instead of using the API.

**Other datasets worth knowing about**, if you want to swap later:
- *Kaggle "Car License Plate Detection" (andrewmvd)* — small (433 images), Pascal
  VOC XML annotations, good for a very fast smoke test but needs conversion to YOLO format.
- *Large License Plate Dataset / Open Images "Vehicle registration plate" class* —
  tens of thousands of images if you want to scale up beyond this notebook.

In [ ]:
from roboflow import Roboflow

# Get a free key at https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
dataset = project.version(4).download("yolov8")

DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

In [ ]:
with open(f"{DATASET_DIR}/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

print("Classes:", data_cfg["names"])
for split in ["train", "valid", "test"]:
    img_dir = f"{DATASET_DIR}/{split}/images"
    if os.path.exists(img_dir):
        print(f"{split}: {len(os.listdir(img_dir))} images")

## 3. Dataset exploration

Quick sanity check: draw the YOLO-format label boxes over a few training images
so you can confirm the annotations line up before spending compute on training.

In [ ]:
def yolo_to_pixel(box, img_w, img_h):
    cls, xc, yc, w, h = box
    x1 = int((xc - w / 2) * img_w)
    y1 = int((yc - h / 2) * img_h)
    x2 = int((xc + w / 2) * img_w)
    y2 = int((yc + h / 2) * img_h)
    return int(cls), x1, y1, x2, y2


def show_samples(split="train", n=6):
    img_dir = f"{DATASET_DIR}/{split}/images"
    lbl_dir = f"{DATASET_DIR}/{split}/labels"
    img_files = sorted(os.listdir(img_dir))[:n]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, fname in zip(axes.flatten(), img_files):
        img = cv2.cvtColor(cv2.imread(f"{img_dir}/{fname}"), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = f"{lbl_dir}/{Path(fname).stem}.txt"
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    box = list(map(float, line.split()))
                    _, x1, y1, x2, y2 = yolo_to_pixel(box, w, h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

        ax.imshow(img)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_samples("train")

## 4. Training

`yolov8s.pt` (small) is a good accuracy/speed balance for this task. Swap to
`yolov8n.pt` if you want faster iteration, or `yolov8m.pt` / `yolov8l.pt` if
you have GPU budget and want to push accuracy further.

`patience=15` stops training early if validation mAP hasn't improved in 15
epochs, so raising `epochs` costs little if the model converges sooner.

In [ ]:
model = YOLO("yolov8s.pt")

results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if torch.cuda.is_available() else "cpu",
    project="license_plate_runs",
    name="yolov8s_lp",
    plots=True,
)

In [ ]:
display(IPyImage(filename="license_plate_runs/yolov8s_lp/results.png", width=900))
display(IPyImage(filename="license_plate_runs/yolov8s_lp/confusion_matrix.png", width=600))

## 5. Validation

In [ ]:
best_model = YOLO("license_plate_runs/yolov8s_lp/weights/best.pt")
metrics = best_model.val(data=f"{DATASET_DIR}/data.yaml")

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

## 6. Inference (detection only)

In [ ]:
test_images = sorted(glob.glob(f"{DATASET_DIR}/test/images/*.jpg"))[:6]
if not test_images:
    test_images = sorted(glob.glob(f"{DATASET_DIR}/valid/images/*.jpg"))[:6]

det_results = best_model.predict(test_images, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flatten(), det_results):
    im = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(im)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Cropping detected plates

YOLO only gives us *where* the plate is. To read the characters we crop that
region out of the full image and hand the crop to OCR.

In [ ]:
def crop_plates(image_path, model, conf=0.25):
    img = cv2.imread(image_path)
    results = model.predict(image_path, conf=conf, verbose=False)
    crops, boxes = [], []
    for r in results:
        for box in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            crops.append(img[y1:y2, x1:x2])
            boxes.append((x1, y1, x2, y2))
    return crops, boxes


crops, boxes = crop_plates(test_images[0], best_model)
print(f"Found {len(crops)} plate(s) in {test_images[0]}")
if crops:
    plt.imshow(cv2.cvtColor(crops[0], cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 8. OCR preprocessing

Plate crops are small and often low-contrast, which is where most OCR errors
come from — not from EasyOCR itself. Upscaling, denoising, and thresholding
before OCR consistently improves character accuracy more than swapping OCR
engines does.

In [ ]:
def preprocess_plate(crop, upscale=2, use_clahe=True):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.resize(gray, (w * upscale, h * upscale), interpolation=cv2.INTER_CUBIC)

    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)

    gray = cv2.bilateralFilter(gray, 11, 17, 17)
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )
    return thresh


if crops:
    processed = preprocess_plate(crops[0])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(crops[0], cv2.COLOR_BGR2RGB))
    axes[0].set_title("Raw crop")
    axes[0].axis("off")
    axes[1].imshow(processed, cmap="gray")
    axes[1].set_title("Preprocessed")
    axes[1].axis("off")
    plt.show()

## 9. OCR with EasyOCR

EasyOCR is the standard pairing with YOLOv8 for this task: deep-learning based
(handles noisy/blurred crops far better than Tesseract), minimal setup, and
GPU-accelerated in Colab.

In [ ]:
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

In [ ]:
def clean_plate_text(text):
    return re.sub(r"[^A-Z0-9]", "", text.upper())


def ocr_plate(crop, reader, preprocess=True):
    image_for_ocr = preprocess_plate(crop) if preprocess else crop
    result = reader.readtext(image_for_ocr, detail=0, paragraph=False)
    return clean_plate_text("".join(result))


if crops:
    print("Detected text:", ocr_plate(crops[0], reader))

## 10. End-to-end pipeline

Detect -> crop -> preprocess -> OCR -> draw box + text, wrapped into one call.

In [ ]:
def recognize_license_plate(image_path, detector, reader, conf=0.25, show=True):
    img = cv2.imread(image_path)
    results = detector.predict(image_path, conf=conf, verbose=False)

    plates = []
    for r in results:
        for box in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            crop = img[y1:y2, x1:x2]
            text = ocr_plate(crop, reader)
            plates.append({"bbox": (x1, y1, x2, y2), "text": text})
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img, text, (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    if show:
        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()

    return plates


for img_path in test_images[:3]:
    print(img_path)
    recognize_license_plate(img_path, best_model, reader)

## 11. Batch processing + CSV export

In [ ]:
records = []
for img_path in test_images:
    for p in recognize_license_plate(img_path, best_model, reader, show=False):
        records.append({
            "image": os.path.basename(img_path),
            "plate_text": p["text"],
            "bbox": p["bbox"],
        })

df = pd.DataFrame(records)
df.to_csv("plate_recognition_results.csv", index=False)
df.head(10)

## 12. Bonus: video pipeline

Runs detection every `skip_frames` frames (OCR is the slow part) and holds
the last reading in between, so the overlay doesn't flicker every frame.

In [ ]:
def process_video(video_path, output_path, detector, reader, conf=0.25, skip_frames=2):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    frame_idx = 0
    last_plates = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % skip_frames == 0:
            results = detector.predict(frame, conf=conf, verbose=False)
            last_plates = []
            for r in results:
                for box in r.boxes.xyxy.cpu().numpy():
                    x1, y1, x2, y2 = map(int, box)
                    crop = frame[y1:y2, x1:x2]
                    text = ocr_plate(crop, reader)
                    last_plates.append((x1, y1, x2, y2, text))

        for (x1, y1, x2, y2, text) in last_plates:
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(frame, text, (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print("Saved:", output_path)


# Example (uncomment once you have a video file uploaded):
# process_video("input.mp4", "output.mp4", best_model, reader)

## 13. Improving accuracy further

**Detection side**
- Train longer / on `yolov8m.pt` if mAP50 is below ~0.90 on validation.
- Raise `imgsz` to 800-960 if plates are small relative to the full frame
  (small-object detection improves with resolution).
- Run `model.tune()` (Ultralytics' built-in hyperparameter search) if you
  have compute budget to spend.
- Use `augment=True` in `predict()` for test-time augmentation — slower but
  more robust at inference.

**OCR side**
- The preprocessing in section 8 (upscale + CLAHE + adaptive threshold) is
  usually worth more than switching OCR engines. Try disabling CLAHE
  (`use_clahe=False`) if your plates are already high-contrast — it can
  sometimes over-sharpen and hurt clean images.
- PaddleOCR is a strong alternative to EasyOCR, often slightly more accurate
  on structured text, at the cost of a heavier install.
- If you know the target region's plate format, validate/correct OCR output
  against it — the two commonly-confused character pairs are `O/0`, `I/1`,
  `S/5`, `B/8`, `Z/2`.
- For video, take a majority vote of the OCR reading across several frames of
  the same vehicle instead of trusting a single frame.

In [ ]:
# Character-confusion correction against a known plate format.
CONFUSABLE_PAIRS = {"O": "0", "0": "O", "I": "1", "1": "I",
                     "S": "5", "5": "S", "B": "8", "8": "B", "Z": "2", "2": "Z"}


def try_format_correction(text, pattern):
    # If text doesn't match pattern, try swapping one confusable character
    # at a time and return the first swap that produces a match.
    if re.match(pattern, text):
        return text
    for i, ch in enumerate(text):
        if ch in CONFUSABLE_PAIRS:
            candidate = text[:i] + CONFUSABLE_PAIRS[ch] + text[i + 1:]
            if re.match(pattern, candidate):
                return candidate
    return text


PLATE_PATTERNS = {
    "generic": r"^[A-Z0-9]{5,10}$",
    "india": r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,2}[0-9]{4}$",
}

# Example:
# try_format_correction("MH12AB1Z34", PLATE_PATTERNS["india"])

## 14. Export & persist the model

Colab sessions are ephemeral — export weights and copy them to Drive so a
disconnect doesn't cost you the trained model.

In [ ]:
best_model.export(format="onnx")

# Optional: persist results to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r license_plate_runs /content/drive/MyDrive/license_plate_runs

### Next steps

- Wrapping `recognize_license_plate` in a FastAPI endpoint (image in, JSON
  plate text + bbox out) is a natural way to turn this into a deployable
  service rather than a notebook-only demo.
- If accuracy on your own images is lower than on the test set, the fastest
  fix is usually adding ~100-200 manually-annotated images from your actual
  use case and fine-tuning `best.pt` further, rather than tuning
  hyperparameters on the public dataset.